In [ ]:
import numpy as np
from dataclasses import dataclass
from typing import Optional
try:
    from scipy.integrate import quad
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False
from astropy import units as u
from astropy import constants as const

C_KMS = const.c.to(u.km/u.s).value

@dataclass
class CosmoParams:
    H0: float = 70.0            # [km/s/Mpc]
    Om0: float = 0.3            # Ω_m
    Ok0: float = 0.0            # Ω_k
    Or0: float = 0.0            # Ω_r (可忽略时设 0)
    # 暗能量状态方程：二选一
    w: Optional[float] = None      # 常数 w
    w0: float = -1.0            # CPL: w(a)=w0+wa(1-a)
    wa: float = 0.0

    def Ode0(self) -> float:
        return 1.0 - self.Om0 - self.Ok0 - self.Or0

def _rho_de_factor(z: float, cp: CosmoParams) -> float:
    """ρ_DE(z)/ρ_DE(0) 的无量纲演化因子。"""
    if cp.w is not None:  # 常数 w
        return (1.0 + z)**(3.0 * (1.0 + cp.w))
    # CPL: w(a)=w0+wa(1-a) => ρ_DE ∝ (1+z)^{3(1+w0+wa)} exp[-3 wa z/(1+z)]
    return (1.0 + z)**(3.0 * (1.0 + cp.w0 + cp.wa)) * np.exp(-3.0 * cp.wa * z / (1.0 + z))

def Ez(z: float, cp: CosmoParams) -> float:
    """E(z) = H(z)/H0."""
    de = cp.Ode0() * _rho_de_factor(z, cp)
    return np.sqrt(cp.Om0 * (1.0 + z)**3 + cp.Or0 * (1.0 + z)**4 + cp.Ok0 * (1.0 + z)**2 + de)

def _chi_quad(z: float, cp: CosmoParams) -> float:
    """无量纲共动径向距离 χ(z) = ∫_0^z dz'/E(z')."""
    if _HAS_SCIPY:
        val, _ = quad(lambda zp: 1.0 / Ez(zp, cp), 0.0, float(z), epsrel=1e-8, epsabs=0.0, limit=256)
        return val
    # 简单梯形积分后备方案
    zgrid = np.linspace(0.0, float(z), 4097)
    return np.trapz(1.0 / np.vectorize(Ez)(zgrid, cp), zgrid)

def _Sk(x: float, Ok0: float) -> float:
    """曲率函数 S_k(x)。注意这里的 x 是 √|Ω_k| χ。"""
    if abs(Ok0) < 1e-12:
        return x
    if Ok0 > 0.0:  # open, sinh
        return np.sinh(x)
    else:          # closed, sin
        return np.sin(x)

def D_M(z: float, cp: CosmoParams) -> float:
    """横向共动距离 D_M(z) [Mpc]."""
    chi = _chi_quad(z, cp)
    if abs(cp.Ok0) < 1e-12:
        return (C_KMS / cp.H0) * chi
    sqrtOk = np.sqrt(abs(cp.Ok0))
    return (C_KMS / cp.H0) * _Sk(sqrtOk * chi, cp.Ok0) / sqrtOk

def D_A(z: float, cp: CosmoParams) -> float:
    """角径距 D_A(0→z) [Mpc]."""
    return D_M(z, cp) / (1.0 + z)

def D_A12(z1: float, z2: float, cp: CosmoParams) -> float:
    """两红移间角径距 D_A(z1→z2) [Mpc], 需 z2>z1。"""
    if not (z2 > z1):
        raise ValueError("Require z2 > z1 for D_A12.")
    chi1 = _chi_quad(z1, cp)
    chi2 = _chi_quad(z2, cp)
    dchi = chi2 - chi1
    if abs(cp.Ok0) < 1e-12:
        DM12 = (C_KMS / cp.H0) * dchi
    else:
        sqrtOk = np.sqrt(abs(cp.Ok0))
        DM12 = (C_KMS / cp.H0) * _Sk(sqrtOk * dchi, cp.Ok0) / sqrtOk
    return DM12 / (1.0 + z2)

# ---- 封装常用距离 ----
def Dl(zl: float, cp: CosmoParams) -> float: return D_A(zl, cp)
def Ds(zs: float, cp: CosmoParams) -> float: return D_A(zs, cp)
def Dls(zl: float, zs: float, cp: CosmoParams) -> float: return D_A12(zl, zs, cp)

# =========================
# 1) 星系–星系透镜：R = Ds / Dls
# =========================
def galaxy_lens_distance_ratio(zl: float, zs: float, *,
                               H0: float, omega_m: float,
                               omega_k: float = 0.0, omega_r: float = 0.0,
                               w: Optional[float] = None, w0: float = -1.0, wa: float = 0.0) -> float:
    """
    返回 R = D_s / D_ls （无量纲），仅含宇宙学部分。
    注：真实建模还需乘以结构因子 F(γ, β_ani, θ_ap/θ_E) 才能与(θ_E, σ_ap)组合。
    """
    cp = CosmoParams(H0=H0, Om0=omega_m, Ok0=omega_k, Or0=omega_r, w=w, w0=w0, wa=wa)
    return Ds(zs, cp) / Dls(zl, zs, cp)

# =========================
# 2) 时延 AGN：D_dt = (1+zl) Dl Ds / Dls
# =========================
def time_delay_distance(zl: float, zs: float, *,
                        H0: float, omega_m: float,
                        omega_k: float = 0.0, omega_r: float = 0.0,
                        w: Optional[float] = None, w0: float = -1.0, wa: float = 0.0) -> float:
    """
    返回时延距离 D_Δt [Mpc]： (1+zl) * Dl * Ds / Dls
    注意：观测上 D_Δt 还会被外会聚 κ_ext 等效缩放，建模需单独处理。
    """
    cp = CosmoParams(H0=H0, Om0=omega_m, Ok0=omega_k, Or0=omega_r, w=w, w0=w0, wa=wa)
    return (1.0 + zl) * Dl(zl, cp) * Ds(zs, cp) / Dls(zl, zs, cp)

# =========================
# 3) 双源透镜：β_DSP = (D_ls2/D_s2) / (D_ls1/D_s1)
# =========================
def double_source_beta(zl: float, zs1: float, zs2: float, *,
                       H0: float, omega_m: float,
                       omega_k: float = 0.0, omega_r: float = 0.0,
                       w: Optional[float] = None, w0: float = -1.0, wa: float = 0.0) -> float:
    """
    返回双源透镜的宇宙学量 β_DSP （无量纲）。
    要求 zl < zs1 < zs2。
    """
    if not (zl < zs1 < zs2):
        raise ValueError("Require zl < zs1 < zs2 for double-source lensing.")
    cp = CosmoParams(H0=H0, Om0=omega_m, Ok0=omega_k, Or0=omega_r, w=w, w0=w0, wa=wa)
    num = Dls(zl, zs2, cp) / Ds(zs2, cp)
    den = Dls(zl, zs1, cp) / Ds(zs1, cp)
    return num / den

import numpy as np

def _as_float(x):
    # 确保传给积分器的是标量 float
    return float(np.asarray(x))

def galaxy_lens_distance_ratio_vec(
    zl, zs, *, H0, omega_m, omega_k=0.0, omega_r=0.0,
    w=None, w0=-1.0, wa=0.0
):
    """
    向量化版本：返回 R = Ds/Dls（与 zl、zs 广播后的形状一致）。
    不满足 zs>zl 的元素返回 np.nan。
    """
    cp = CosmoParams(H0=H0, Om0=omega_m, Ok0=omega_k, Or0=omega_r, w=w, w0=w0, wa=wa)
    zlA, zsA = np.broadcast_arrays(np.asarray(zl, float), np.asarray(zs, float))
    out = np.empty_like(zlA, dtype=float)
    it = np.nditer([zlA, zsA, out], flags=['multi_index'],
                   op_flags=[['readonly'], ['readonly'], ['writeonly']])
    for zl_i, zs_i, o in it:
        zl_f, zs_f = _as_float(zl_i), _as_float(zs_i)
        if zs_f <= zl_f:
            o[...] = np.nan
        else:
            # o[...] = Ds(zs_f, cp) / Dls(zl_f, zs_f, cp)
            o[...] = Dls(zl_f, zs_f, cp) / Ds(zs_f, cp)
    return out

def time_delay_distance_vec(
    zl, zs, *, H0, omega_m, omega_k=0.0, omega_r=0.0,
    w=None, w0=-1.0, wa=0.0
):
    """
    向量化版本：返回 D_dt [Mpc] = (1+zl) * Dl * Ds / Dls。
    不满足 zs>zl 的元素返回 np.nan。
    """
    cp = CosmoParams(H0=H0, Om0=omega_m, Ok0=omega_k, Or0=omega_r, w=w, w0=w0, wa=wa)
    zlA, zsA = np.broadcast_arrays(np.asarray(zl, float), np.asarray(zs, float))
    out = np.empty_like(zlA, dtype=float)
    it = np.nditer([zlA, zsA, out], flags=['multi_index'],
                   op_flags=[['readonly'], ['readonly'], ['writeonly']])
    for zl_i, zs_i, o in it:
        zl_f, zs_f = _as_float(zl_i), _as_float(zs_i)
        if zs_f <= zl_f:
            o[...] = np.nan
        else:
            o[...] = (1.0 + zl_f) * Dl(zl_f, cp) * Ds(zs_f, cp) / Dls(zl_f, zs_f, cp)
    return out

def double_source_beta_vec(
    zl, zs1, zs2, *, H0, omega_m, omega_k=0.0, omega_r=0.0,
    w=None, w0=-1.0, wa=0.0
):
    """
    向量化版本：返回 β_DSP = (D_ls2/D_s2)/(D_ls1/D_s1)。
    仅当 zl < zs1 < zs2 时有效，否则该元素 np.nan。
    """
    cp = CosmoParams(H0=H0, Om0=omega_m, Ok0=omega_k, Or0=omega_r, w=w, w0=w0, wa=wa)
    zlA, z1A, z2A = np.broadcast_arrays(np.asarray(zl, float),
                                        np.asarray(zs1, float),
                                        np.asarray(zs2, float))
    out = np.empty_like(zlA, dtype=float)
    it = np.nditer([zlA, z1A, z2A, out], flags=['multi_index'],
                   op_flags=[['readonly'], ['readonly'], ['readonly'], ['writeonly']])
    for zl_i, z1_i, z2_i, o in it:
        zl_f, z1_f, z2_f = _as_float(zl_i), _as_float(z1_i), _as_float(z2_i)
        if not (zl_f < z1_f < z2_f):
            o[...] = np.nan
        else:
            num = Dls(zl_f, z2_f, cp) / Ds(z2_f, cp)
            den = Dls(zl_f, z1_f, cp) / Ds(z1_f, cp)
            o[...] = num / den
    return out



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List

# ====== 1) 时延距离标量 ======
def Ddt_scalar(zl: float, zs: float, cosmo: Dict) -> float:
    return time_delay_distance(float(zl), float(zs), **cosmo)

def R_scalar(zl: float, zs: float, cosmo: Dict) -> float:
    # 只取 galaxy lens 的宇宙学部分：R = Ds/Dls
    return galaxy_lens_distance_ratio(float(zl), float(zs), **cosmo)

def beta_scalar(zl: float, zs1: float, zs2: float, cosmo: Dict) -> float:
    """β_DSP(zl; zs1, zs2) = (D_ls2/D_s2) / (D_ls1/D_s1)."""
    return double_source_beta(float(zl), float(zs1), float(zs2), **cosmo)

def dlnR_dp(zl: float, zs: float, cosmo: dict, p: str,
            eps_rel: float = 1e-5, eps_abs: float = 1e-5) -> float:
    base = R_scalar(zl, zs, cosmo)
    if not np.isfinite(base) or base <= 0:
        return np.nan

    val = cosmo.get(p, None)
    # step choice
    if (val is None) or (abs(val) < 1e-12) or (p in ["w0", "wa", "omega_k"]):
        dp = eps_abs if p not in ["omega_m", "H0"] else max(eps_abs, 1e-6)
        p_minus = (val - dp) if (val is not None) else -dp
        p_plus  = (val + dp) if (val is not None) else  dp
    else:
        dp = abs(val) * eps_rel
        p_minus, p_plus = val - dp, val + dp

    # physical guards
    if p == "omega_m":
        p_minus = max(1e-6, p_minus); p_plus = max(2e-6, p_plus)
    if p == "omega_k":
        p_minus = np.clip(p_minus, -0.5, 0.5); p_plus = np.clip(p_plus, -0.5, 0.5)
    if p == "H0":
        p_minus = max(1.0, p_minus); p_plus = max(2.0, p_plus)

    c_m = cosmo.copy(); c_m[p] = p_minus
    c_p = cosmo.copy(); c_p[p] = p_plus
    Rm = R_scalar(zl, zs, c_m)
    Rp = R_scalar(zl, zs, c_p)
    if (Rm <= 0) or (Rp <= 0) or (not np.isfinite(Rm)) or (not np.isfinite(Rp)):
        return np.nan
    return float((np.log(Rp) - np.log(Rm)) / (p_plus - p_minus))

# ====== 2) 数值导数：∂ ln D_dt / ∂ p ======
def dlnDdt_dp(zl: float, zs: float, cosmo: Dict, p: str,
              eps_rel: float = 1e-5, eps_abs: float = 1e-5) -> float:
    cosmo_p = cosmo.copy()
    base = Ddt_scalar(zl, zs, cosmo_p)
    if not np.isfinite(base) or base <= 0:
        return np.nan

    val = cosmo_p.get(p, None)
    # 步长
    if (val is None) or (abs(val) < 1e-12) or (p in ["w0", "wa", "omega_k"]):
        dp = eps_abs if p not in ["omega_m", "H0"] else max(eps_abs, 1e-6)
        p_minus, p_plus = (val - dp if val is not None else -dp), (val + dp if val is not None else dp)
    else:
        dp = abs(val) * eps_rel
        p_minus, p_plus = val - dp, val + dp

    # 物理边界
    if p == "omega_m":
        p_minus = max(1e-6, p_minus); p_plus = max(2e-6, p_plus)
    if p == "omega_k":
        p_minus = np.clip(p_minus, -0.5, 0.5)
        p_plus  = np.clip(p_plus,  -0.5, 0.5)
    if p == "H0":
        p_minus = max(1.0, p_minus); p_plus = max(2.0, p_plus)

    # 中心差分
    cosmo_m = cosmo.copy();  cosmo_m[p]  = p_minus
    cosmo_p2 = cosmo.copy(); cosmo_p2[p] = p_plus
    Dm = Ddt_scalar(zl, zs, cosmo_m)
    Dp = Ddt_scalar(zl, zs, cosmo_p2)
    if (Dm <= 0) or (Dp <= 0) or (not np.isfinite(Dm)) or (not np.isfinite(Dp)):
        return np.nan

    return float((np.log(Dp) - np.log(Dm)) / (p_plus - p_minus))

def dlnbeta_dp(zl: float, zs1: float, zs2: float, cosmo: Dict, p: str,
               eps_rel: float = 1e-5, eps_abs: float = 1e-5) -> float:
    """
    对参数 p 求 ∂ ln β_DSP 的数值导数。
    - 对可能为 0/有号的参数（w0, wa, omega_k）用绝对步长；其余用相对步长。
    - 注意 β_DSP 与 H0 完全相消；若 p='H0'，导数应≈0（数值误差级）。
    """
    base = beta_scalar(zl, zs1, zs2, cosmo)
    if not np.isfinite(base) or base <= 0:
        return np.nan

    val = cosmo.get(p, None)
    if (val is None) or (abs(val) < 1e-12) or (p in ["w0", "wa", "omega_k"]):
        dp = eps_abs if p not in ["omega_m", "H0"] else max(eps_abs, 1e-6)
        p_minus = (val - dp) if (val is not None) else -dp
        p_plus  = (val + dp) if (val is not None) else  dp
    else:
        dp = abs(val) * eps_rel
        p_minus, p_plus = val - dp, val + dp

    # 物理边界
    if p == "omega_m":
        # p_minus = max(1e-6, p_minus); p_plus = max(1e-6, p_plus)
        p_minus = np.clip(p_minus, -0.5, 0.5); p_plus = np.clip(p_plus, -0.5, 0.5)
    if p == "omega_k":
        p_minus = np.clip(p_minus, -0.5, 0.5); p_plus = np.clip(p_plus, -0.5, 0.5)
    if p == "H0":
        p_minus = max(1.0, p_minus); p_plus = max(2.0, p_plus)

    cosmo_m = cosmo.copy();  cosmo_m[p]  = p_minus
    cosmo_p = cosmo.copy();  cosmo_p[p]  = p_plus

    bm = beta_scalar(zl, zs1, zs2, cosmo_m)
    bp = beta_scalar(zl, zs1, zs2, cosmo_p)
    if (bm <= 0) or (bp <= 0) or (not np.isfinite(bm)) or (not np.isfinite(bp)):
        return np.nan

    return float((np.log(bp) - np.log(bm)) / (p_plus - p_minus))

# ====== helper：确保 H0 在参数列表里，并返回去重后的顺序表 ======
def _with_H0(params: List[str]) -> List[str]:
    seen = set()
    out = []
    for p in list(params) + ["H0"]:
        if p not in seen:
            out.append(p); seen.add(p)
    return out

# ====== 3) Fisher：单点 & 累积（始终把 H0 纳入）======
def fisher_single(zl: float, zs: float, cosmo: Dict, params: List[str], sigma_frac: float):
    """
    返回 (F, params_eff)：
      - F: (len(params_eff) x len(params_eff)) Fisher 矩阵
      - params_eff: 实际使用的参数顺序（原 params + 'H0'，去重）
    """
    params_eff = _with_H0(params)
    J = np.array([dlnDdt_dp(zl, zs, cosmo, p) for p in params_eff], dtype=float)  # ∂ln D_dt/∂p
    if not np.all(np.isfinite(J)):
        return np.zeros((len(params_eff), len(params_eff))), params_eff
    F = np.outer(J, J) / (sigma_frac**2)
    return F, params_eff

def fisher_cumulative(zl_arr: np.ndarray, zs: float, cosmo: Dict, params: List[str], sigma_frac: float):
    """
    把一个 z_s 下所有 z_l 的 Fisher 相加。
    返回 (F, params_eff)。
    """
    params_eff = _with_H0(params)
    F = np.zeros((len(params_eff), len(params_eff)), dtype=float)
    for zl in zl_arr:
        if zl <= 0 or zl >= zs:
            continue
        F_k, _ = fisher_single(float(zl), float(zs), cosmo, params_eff, sigma_frac)
        F += F_k
    return F, params_eff

# ====== 4) zl 网格 ======
def zl_grid_for_zs(zs: float, dz: float = 0.004) -> np.ndarray:
    n = max(1, int(np.floor(zs / dz)))
    eps = 1e-6 * max(1.0, zs)
    return np.linspace(eps, zs - eps, n)



In [ ]:
def plot_sensitivity_R_w0_wa_with_scatter(
    cosmo_cpl,
    LSSTa_lens=None,                 # expects an object with columns/keys 'zl' and 'zs'
    *,  # grids
    zl_min=0.0, zl_max=2.0, zl_step=0.1,
    zs_min=0.0, zs_max=3.0, zs_step=0.1,
    # sensitivity
    sigma_frac=0.1,
    # styling
    cmap="viridis",
    scatter_color="#6a3d9a",        
    scatter_alpha=0.2,
    scatter_size=6,
    # figure
    figsize=(10, 4.2),
    outputname="contours_R_w0_wa_scatter.png"
):
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.colors import LogNorm
    from matplotlib.ticker import LogFormatterMathtext

    # ---- ensure CPL keys (no 'w') ----
    cosmo = dict(H0=cosmo_cpl.get("H0", 70.0),
                 omega_m=cosmo_cpl.get("omega_m", 0.3),
                 omega_k=cosmo_cpl.get("omega_k", 0.0),
                 w0=cosmo_cpl.get("w0", -1.0),
                 wa=cosmo_cpl.get("wa", 0.0))
    cosmo.pop("w", None)

    params    = ["w0", "wa"]
    param_tex = [r"$\mathcal{D}-w_0$", r"$\mathcal{D}-w_a$"]

    # ---- grids ----
    zl_grid = np.arange(zl_min, zl_max + 1e-12, zl_step)
    zs_grid = np.arange(zs_min, zs_max + 1e-12, zs_step)
    ZL, ZS  = np.meshgrid(zl_grid, zs_grid, indexing="xy")

    # ---- sensitivity helper for R ----
    def _grid_S_R(param):
        S = np.empty_like(ZL, dtype=float); S[:] = np.nan
        for i in range(ZL.shape[0]):      # over z_s
            for j in range(ZL.shape[1]):  # over z_l
                zl, zs = ZL[i, j], ZS[i, j]
                if zs <= zl or zl <= 0.0:
                    continue
                d = dlnR_dp(zl, zs, cosmo, param)
                S[i, j] = abs(d) / max(sigma_frac, 1e-30) if np.isfinite(d) else np.nan
        return np.ma.masked_invalid(S)

    # ---- shared color scale (16 levels, 1e-4 to 1e1) ----
    vmin, vmax = 1e-4, 10**0.2
    levels = np.logspace(-4, 0.2, 12)
    norm   = LogNorm(vmin=vmin, vmax=vmax)

    # ---- figure (leave room on right for shared colorbar) ----
    fig, axes = plt.subplots(1, 2, figsize=figsize, sharex=False, sharey=False)
    axes = np.atleast_1d(axes)
    fig.subplots_adjust(right=0.86)  # free right margin for colorbar

    # in-panel parameter label style
    txt_kw = dict(ha="right", va="bottom", fontsize=14)

    # -------- draw the two panels --------
    for ax, p, ptex in zip(axes, params, param_tex):
        S = _grid_S_R(p)
        ax.contourf(zl_grid, zs_grid, S, levels=levels, norm=norm, cmap=cmap, alpha=0.75, zorder=2)
        ax.contour(zl_grid, zs_grid, S, levels=levels, colors="k", linewidths=0.6, alpha=0.75, norm=norm, zorder=4)

        # scatter overlay (if provided)
        if LSSTa_lens is not None:
            # support dict-like or pandas DataFrame
            zl_s = np.asarray(LSSTa_lens['zl'], dtype=float)
            zs_s = np.asarray(LSSTa_lens['zs'], dtype=float)
            m = (
                np.isfinite(zl_s) & np.isfinite(zs_s) &
                (zl_s >= zl_min) & (zl_s <= zl_max) &
                (zs_s >= zs_min) & (zs_s <= zs_max) &
                (zs_s >  zl_s)
            )
            if np.any(m):
                ax.scatter(
                    zl_s[m], zs_s[m],
                    s=scatter_size, marker='o',
                    facecolors='none', edgecolors=scatter_color,
                    linewidths=0.8, alpha=scatter_alpha,
                    rasterized=True, zorder=3, label="LSSTa lenses"
                )

        ax.set_ylabel(r"$z_s$", fontsize=12)
        ax.set_xlabel(r"$z_l$", fontsize=12)
        ax.grid(alpha=0.2)
        ax.text(0.98, 0.02, ptex, transform=ax.transAxes, **txt_kw)

    # ---- shared colorbar on the right ----
    cax = fig.add_axes([0.88, 0.18, 0.02, 0.64])  # [left, bottom, width, height]
    cb  = fig.colorbar(
        plt.cm.ScalarMappable(norm=norm, cmap=cmap),
        cax=cax, ticks=levels, format=LogFormatterMathtext()
    )
    # cb.set_label(r"Sensitivity: $|\partial \ln \mathcal{R}/\partial p|\,/\,\sigma_{\ln \mathcal{R}}$")

    plt.tight_layout(rect=[0.04, 0.04, 0.86, 0.98])
    plt.savefig(outputname, dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
# "/home/astrodust/SG/sgl_cosmo/GLS_cosmo01/Data/LSSTa_sim_l3.fits"

from astropy.table import Table

LSSTa_lens = Table.read(r"/home/geng/Codes/sgl_cosmo/GLS_cosmo01/Data/LSSTa_sim_l3.fits")

# Example:
cosmo_cpl  = dict(H0=70.0, omega_m=0.3, omega_k=0.0, w0=-0.95, wa=0.0)

# LSSTa_lens should be a DataFrame or dict-like with columns 'zl' and 'zs'
# e.g., LSSTa_lens = {"zl": zl_array, "zs": zs_array}

plot_sensitivity_R_w0_wa_with_scatter(
    cosmo_cpl,
    LSSTa_lens=LSSTa_lens,
    zl_min=0.0, zl_max=2.0, zl_step=0.01,
    zs_min=0.0, zs_max=3.0, zs_step=0.01,
    sigma_frac=0.1,
    scatter_alpha=0.2,
    scatter_size=14,
    scatter_color="royalblue",  # purple
    outputname="contours_R_w0_wa+scatter.png"
)
